# Etterforskning

Vi har en stor ferdiglaget graf, med titusenvis av noder.  Vi skal bruke Neo4J til å lete etter kriminalitet.


In [ ]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)

podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

Om dette feiler, forsøk igjen om en liten stund; databasen skal laste ned utvidelser og initalisere seg.  Den feiler om den ikke har nett.

In [1]:
# Få Kontakt
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Kontakt!")

Kontakt!


Vi flytter grafen vi skal arbeide med

In [2]:
%%bash
cp grafer/Komplett.graphml neo4j/import

### ToDo
Hente alle projeeksjoner og slette dem, og slette alle noder.

In [3]:
# Ikke starte med gamle data noe sted
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('VårGraf', false) YIELD graphName"""
)
records, summary, keys = driver.execute_query(
    """match (n) detach delete n""")
print("OK")

OK


In [5]:
# Lese inn grafen vi har bygget tidligere
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("Komplett.graphml", {storeNodeIds: true, readLabels: true})"""
    )
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: Komplett.graphml
	source: file
	format: graphml
	nodes: 56827
	relationships: 103562
	properties: 57646
	time: 1294
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 59ms
	Å konsumere: 1297ms


## Lete etter et esel
Vi ser etter en klikk hvor alle kjenner hverandre.  Vi finner kanskje mange.  Det er interessant om (nesten) alle (eller i det minste flertallet) har uttak av kontanter.
Dette er typisk GDS-mat (algoritmer som kjører på hele grafen, ikke bare på enkeltnoder).

In [6]:
# Først, sjekke at det vi leter faktisk er der (sånn for sikkerhets skyld)
records, summary, keys = driver.execute_query(
  """
    MATCH (N:Person {Esel:True}) return N limit 20
  """
)
print(f"Antall i settet er {len(records)}")

Antall i settet er 20


Vi leter etter en klikk, hvor alle kjenner hverandre.  Vi starter med å hente ut alle personer.  Vi henter altså ikke ut konti og transkaskjoner.  Det er mer enn 100.000 kanter i EPOST-settet så mye "blir igjen".

In [7]:
# Hente personer og kun Kjenner-relasjonen
records, summary, keys = driver.execute_query(
  """CALL 
    gds.graph.project(
      'VårGraf',
      'Person',              // Type node
      {
        Kjenner: {           //  Type på kanter
          type: 'Kjenner',
          orientation: 'UNDIRECTED' // Betrakter dem som om de var uten retning
        }
      }
    )
  YIELD nodeCount, relationshipCount, projectMillis;
  """
)
# Det skal ikke komme data tilbake men info om subgrafen.  Det vil si det kommer
# en dict med info om hvor mange noder og kanter som ble hentet
for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	nodeCount: 113344
	relationshipCount: 1536
	projectMillis: 143
Ressursbruk
	Kjøringen: 59ms
	Å konsumere: 151ms


Nå trenger vi en liste over folk i klikker, slik at vi vet hvor vi skal lete.  SOm vi så tidligere så er klikk en "sterk" datastruktur som ikke så lett oppstår tilfeldig, heller ikke i grafer uten skala.

I virkeligheten ville vi ha forsøkt en rekke verdier og undersøkt settet for hver verdi.  Vi kan imidlertid kortslutte litt

In [9]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.stream('VårGraf')
    YIELD nodeId, coreValue
    WITH gds.util.asNode(nodeId) AS N, coreValue
    WHERE coreValue >= 13  // Verdien vi forsøker
    RETURN N.id, coreValue
    """
)
# Hvordan gikk det
for r in records:
    print(r)
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

<Record N.id='d1d1f5d5-8f9f-4ea1-b0b5-467d0c1ed2c9' coreValue=14>
<Record N.id='61e6182a-7ca8-4260-89ec-4823124ea8dc' coreValue=14>
<Record N.id='765d2c7f-c16d-4187-b0eb-670491cc0111' coreValue=14>
<Record N.id='b18f0a88-b48d-4a2e-8794-6260fa2551dd' coreValue=14>
<Record N.id='1bdd4b2a-1ebd-47f2-a0e3-e9b7e7de7e76' coreValue=14>
<Record N.id='2a65a7fe-4e03-4d13-995d-0e1cd898e06e' coreValue=14>
<Record N.id='ae23e377-d5fe-4b11-a9e0-4d650026cc6b' coreValue=14>
<Record N.id='3b9c6f1c-8610-4930-8ead-76be7c65f3c9' coreValue=14>
<Record N.id='6b7d6cf6-52c1-4012-9c12-ba609c23551a' coreValue=14>
<Record N.id='e2ae71d1-e17c-4be8-abf9-3ff19a5bcb36' coreValue=14>
<Record N.id='f8c2d35a-1cb9-48e4-824b-0d9799588642' coreValue=14>
<Record N.id='83db98ef-a68c-494c-a6e0-347b0b1b416f' coreValue=14>
<Record N.id='8ff9b577-bc6d-4087-869b-33a06074fca4' coreValue=14>
<Record N.id='0a79292b-e3c2-4057-b1bb-18a9a4f6dedd' coreValue=14>
<Record N.id='ce2afbb0-b557-4800-a3c8-d77184063042' coreValue=14>
<Record N.

Vi ser vi trolig har en klikk her.  La oss merke disse nodene, og deretter skrive dem tilbake til databasen slik at vi kan vise dem frem i nettleseren.

In [10]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.stream('VårGraf')
    YIELD nodeId, coreValue
    WITH gds.util.asNode(nodeId) AS N, coreValue
    WHERE coreValue >= 13  // Verdien vi forsøker
    SET N.EselMistanke = 1
    RETURN N.id, N.Esel, N.EselMistanke
    """
)
# Vi ber ikke om noe i return, så dette skal være tomt
for r in records:
    print(r)
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

<Record N.id='d1d1f5d5-8f9f-4ea1-b0b5-467d0c1ed2c9' N.Esel=True N.EselMistanke=1>
<Record N.id='61e6182a-7ca8-4260-89ec-4823124ea8dc' N.Esel=True N.EselMistanke=1>
<Record N.id='765d2c7f-c16d-4187-b0eb-670491cc0111' N.Esel=True N.EselMistanke=1>
<Record N.id='b18f0a88-b48d-4a2e-8794-6260fa2551dd' N.Esel=True N.EselMistanke=1>
<Record N.id='1bdd4b2a-1ebd-47f2-a0e3-e9b7e7de7e76' N.Esel=True N.EselMistanke=1>
<Record N.id='2a65a7fe-4e03-4d13-995d-0e1cd898e06e' N.Esel=True N.EselMistanke=1>
<Record N.id='ae23e377-d5fe-4b11-a9e0-4d650026cc6b' N.Esel=True N.EselMistanke=1>
<Record N.id='3b9c6f1c-8610-4930-8ead-76be7c65f3c9' N.Esel=True N.EselMistanke=1>
<Record N.id='6b7d6cf6-52c1-4012-9c12-ba609c23551a' N.Esel=True N.EselMistanke=1>
<Record N.id='e2ae71d1-e17c-4be8-abf9-3ff19a5bcb36' N.Esel=True N.EselMistanke=1>
<Record N.id='f8c2d35a-1cb9-48e4-824b-0d9799588642' N.Esel=True N.EselMistanke=1>
<Record N.id='83db98ef-a68c-494c-a6e0-347b0b1b416f' N.Esel=True N.EselMistanke=1>
<Record N.id='8f

Vi har merket nodene i projeksjonen, skriv det vi har merket tilbake til databsen (slik at vi kan se på resultatet)

In [11]:
# Så skal vi lete etter en klikk av folk
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.write('VårGraf', {
        writeProperty: 'EselMistanke'
    })
    YIELD nodePropertiesWritten, degeneracy
    """
)
# Vi ber ikke om noe i return, så dette skal være tomt
for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	nodePropertiesWritten: 113344
	degeneracy: 14
Ressursbruk
	Kjøringen: 59ms
	Å konsumere: 386ms


Om alt fungerer som det skal, da vil denne koden i nettleseren finne Eselet for oss:
```
match (n:Person {EselMistanke:1}) return n limit 20
```

In [12]:
# Opprydding
# Fjerne projeksjonen
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('VårGraf', false) YIELD graphName"""
)
# Fjerne merkingne
records, summary, keys = driver.execute_query(
    """
    MATCH (N:Person  {EselMistanke:1})
    REMOVE N.EselMistanke
    RETURN COUNT(N) AS Antall
    """
)
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#


	Antall: 56


## Lete etter en bakmann

Vi leter etter et sett med klikker, som alle har egenskapen at de er knyttet sammen gjennom en Bakmann.  Det vil i praksis si en "stjerne" med bakmannen i "sentrum".

Vi gjør noen antagelser (som vi kan eksperimentere med senere):
- Gjengene er (ekte) klikker;
- Klikkene har minst fem medlemmer (med hundre tusen noder er det "uendelig" mange mindre klikker);
- En Bakmann har (minst) tre klikker han håndterer, og
- Bakmannen er ikke med i klikken, altså at bakmannen ikke "Kjenner" alle medlemmene.


Strategien blir:
1. Lage en projeksjon av alle personer
2. Finne *local cluster coefficient* (*LCC*) for alle noder (vi har hentet inn), og skriv verdien tilbake i databasen.  Vi gikk gjennom *LCC* i den generelle delen;
3. En bakmann vil ha (mange) færre venner (relasjonen Kjenner) enn medlemmer av klikkene han "håndterer".  Derfor henter vi ut Personer med få naboer (lav LCC);
4. Som Person har Bakmannen konto.  Siden vi har antatt at han vil "håndtere" (minst) tre klikker vil ha (minst) fire relasjoner, og
5. Bakmannens venner, derimot, er med i klikker og har høy LCC (minst 0.7)

Altså: etter å ha merket nodene med `LCC` finner vi noder med `LCC < 0.1` men som har naboer med `LCC > 0.7`.  Vi merker dem med `MISTENKT` for inspeksjkon.

In [13]:
# Fjerne projeksjonen om den finnes
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('BakmannGraf', false) YIELD graphName"""
)
# Fjerne merkingne
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#

In [14]:
# Lag projeksjonen
records, summary, keys = driver.execute_query(
"""
  CALL gds.graph.project(
  'BakmannGraf',
  'Person',
  {
    Kjenner: { orientation: 'UNDIRECTED' }  // Fjerne eventuell retning
  }
  )
  YIELD nodeCount, relationshipCount, projectMillis;
""")

for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	nodeCount: 113344
	relationshipCount: 1536
	projectMillis: 28
Ressursbruk
	Kjøringen: 31ms
	Å konsumere: 34ms


In [15]:
# Regn ut LCC og skriv tilbake på hver node (som er hentet inn)
# Det betyr at noder av andre typer enn Person ikke får denne
records, summary, keys = driver.execute_query(
"""    
  CALL gds.localClusteringCoefficient.write('BakmannGraf', {
  writeProperty: 'LCC'
});
""")
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")

Ressursbruk
	Kjøringen: 55ms


In [16]:
# Kjøre punktene 3, 4, og 5 over
records, summary, keys = driver.execute_query(
"""
  MATCH (Mistenkt:Person)
  WHERE Mistenkt.LCC < 0.1  // Ikke alle noder har verdi
  AND count{(Mistenkt)--()} >=4 // Minst fire relasjoner
  MATCH (Mistenkt:Person)--(nabo:Person) // Finn naboene
  WHERE nabo.LCC > 0.7 // ...og bare naboer med mange venner
  WITH Mistenkt, collect(nabo) AS Kandidater
  WHERE size(Kandidater) >= 4   // og bare de som har minst tre klikker
  // Når vi kommer hit, da tror vi at vi har den mistenkte
  SET Mistenkt.MISTENKT = 1
  return Mistenkt
""")
for record in records:
  record_dict = record.data()
  for k in record_dict:
      print(f"\t{k}: {record_dict[k]}")
    #
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	Mistenkt: {'EselMistanke': 5, 'Bakmann': True, 'id': 'b5750588-d72a-41a3-96c2-cae8f620f6e5', 'MISTENKT': 1, 'LCC': 0.0, 'Navn': 'Bakmann'}
	Mistenkt: {'EselMistanke': 5, 'Bakmann': True, 'id': 'b5750588-d72a-41a3-96c2-cae8f620f6e5', 'MISTENKT': 1, 'LCC': 0.0, 'Navn': 'Bakmann'}
Ressursbruk
	Kjøringen: 131ms
	Å konsumere: 226ms


I browseren:
```
MATCH (p:Person {MISTENKT=1})
return p
```


In [17]:
# Opprydding
# Fjerne projeksjonen
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('BakmannGraf', false) YIELD graphName"""
)
# Fjerne merkingne
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Person  {MISTENKT:1})
    REMOVE p.MISTENKT
    RETURN COUNT(p) AS Antall
    """
)
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#
# Fjerne merkingne
records, summary, keys = driver.execute_query(
    """
    MATCH (p:Person)
    WHERE p.LCC > 0
    REMOVE p.LCC
    RETURN COUNT(p) AS Antall
    """
)
for record in records:
    record_dict = record.data()
    for k in record_dict:
        print(f"\t{k}: {record_dict[k]}")
    #
#


	Antall: 2
	Antall: 134


## Lete etter deling av utbytte

Har ikke funnet noen mæte å identifisere dette mønsteret på, uten å legge til grunn at jeg allerede vet svaret.